Load logic for `adwm_wh.gold.DimEmployee`.

Business grain:

* One row per `EmployeeID`
* `EmployeeID` is the business key used by the merge
* Employee, person, and current department inputs are reduced to the best row per employee before merge comparison

Source tables:

* `adwm_wh.silver.employee`
* `adwm_wh.silver.person`
* `adwm_wh.silver.employeedepartmenthistory`
* `adwm_wh.silver.department`

Notebook contents:

* Upsert logic for the gold dimension
* Validation summary comparing total rows to distinct `EmployeeID`
* Validation detail query that lists duplicate `EmployeeID` values only if any exist

Operational note:

* `modified_date` is refreshed only when tracked employee attributes change

In [0]:
%sql
WITH employee_dedup AS (
    SELECT
        BusinessEntityID,
        JobTitle,
        Gender,
        HireDate,
        CurrentFlag
    FROM (
        SELECT
            BusinessEntityID,
            JobTitle,
            Gender,
            HireDate,
            CurrentFlag,
            ROW_NUMBER() OVER (
                PARTITION BY BusinessEntityID
                ORDER BY CASE
                    WHEN upper(coalesce(JobTitle, '')) = 'UNKNOWN' THEN 1
                    ELSE 0
                END,
                CASE
                    WHEN upper(coalesce(Gender, '')) = 'UNKNOWN' THEN 1
                    ELSE 0
                END,
                CASE
                    WHEN HireDate IS NULL THEN 1
                    ELSE 0
                END,
                CASE
                    WHEN coalesce(CurrentFlag, false) THEN 0
                    ELSE 1
                END,
                JobTitle,
                Gender,
                HireDate DESC
            ) AS rn
        FROM adwm_wh.silver.employee
    ) e
    WHERE rn = 1
),
person_dedup AS (
    SELECT
        BusinessEntityID,
        FirstName,
        LastName
    FROM (
        SELECT
            BusinessEntityID,
            FirstName,
            LastName,
            ROW_NUMBER() OVER (
                PARTITION BY BusinessEntityID
                ORDER BY CASE
                    WHEN upper(coalesce(FirstName, '')) = 'UNKNOWN'
                     AND upper(coalesce(LastName, '')) = 'UNKNOWN' THEN 1
                    ELSE 0
                END,
                FirstName,
                LastName
            ) AS rn
        FROM adwm_wh.silver.person
    ) p
    WHERE rn = 1
),
current_department AS (
    SELECT
        edh.BusinessEntityID,
        d.Name AS DepartmentName,
        d.GroupName AS DepartmentGroup,
        ROW_NUMBER() OVER (
            PARTITION BY edh.BusinessEntityID
            ORDER BY edh.StartDate DESC, edh.DepartmentID DESC
        ) AS rn
    FROM adwm_wh.silver.employeedepartmenthistory edh
    INNER JOIN adwm_wh.silver.department d
        ON d.DepartmentID = edh.DepartmentID
    WHERE edh.EndDate IS NULL
),
employee_source AS (
    SELECT
        e.BusinessEntityID AS EmployeeID,
        trim(concat_ws(' ', p.FirstName, p.LastName)) AS FullName,
        e.JobTitle,
        e.Gender,
        e.HireDate,
        dept.DepartmentName,
        dept.DepartmentGroup,
        coalesce(e.CurrentFlag, true) AS IsActive,
        sha2(
            concat_ws(
                '||',
                coalesce(trim(concat_ws(' ', p.FirstName, p.LastName)), ''),
                coalesce(e.JobTitle, ''),
                coalesce(e.Gender, ''),
                coalesce(cast(e.HireDate AS STRING), ''),
                coalesce(dept.DepartmentName, ''),
                coalesce(dept.DepartmentGroup, ''),
                coalesce(cast(coalesce(e.CurrentFlag, true) AS STRING), '')
            ),
            256
        ) AS row_hash
    FROM employee_dedup e
    INNER JOIN person_dedup p
        ON p.BusinessEntityID = e.BusinessEntityID
    LEFT JOIN current_department dept
        ON dept.BusinessEntityID = e.BusinessEntityID
       AND dept.rn = 1
)
MERGE INTO adwm_wh.gold.DimEmployee AS target
USING employee_source AS source
ON target.EmployeeID = source.EmployeeID
WHEN MATCHED AND sha2(
    concat_ws(
        '||',
        coalesce(target.FullName, ''),
        coalesce(target.JobTitle, ''),
        coalesce(target.Gender, ''),
        coalesce(cast(target.HireDate AS STRING), ''),
        coalesce(target.DepartmentName, ''),
        coalesce(target.DepartmentGroup, ''),
        coalesce(cast(target.IsActive AS STRING), '')
    ),
    256
) <> source.row_hash THEN UPDATE SET
    target.FullName = source.FullName,
    target.JobTitle = source.JobTitle,
    target.Gender = source.Gender,
    target.HireDate = source.HireDate,
    target.DepartmentName = source.DepartmentName,
    target.DepartmentGroup = source.DepartmentGroup,
    target.IsActive = source.IsActive,
    target.modified_date = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    EmployeeID,
    FullName,
    JobTitle,
    Gender,
    HireDate,
    DepartmentName,
    DepartmentGroup,
    IsActive,
    modified_date
)
VALUES (
    source.EmployeeID,
    source.FullName,
    source.JobTitle,
    source.Gender,
    source.HireDate,
    source.DepartmentName,
    source.DepartmentGroup,
    source.IsActive,
    current_timestamp()
);

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT EmployeeID) AS distinct_employee_ids,
    COUNT(*) - COUNT(DISTINCT EmployeeID) AS duplicate_row_count
FROM adwm_wh.gold.DimEmployee;

In [0]:
%sql
SELECT
    EmployeeID,
    COUNT(*) AS row_count,
    MIN(modified_date) AS first_modified_date,
    MAX(modified_date) AS last_modified_date
FROM adwm_wh.gold.DimEmployee
GROUP BY EmployeeID
HAVING COUNT(*) > 1
ORDER BY row_count DESC, EmployeeID;